# Data Quality Drift Detection: Rediscovering the PSI and Kolmogorov-Smirnov Standard

## **Table of Contents**

- [1 - Business Objective](#1-business-objective)
  - [1.1 - Overview](#11-overview)
  - [1.2 - Business Objective Statement](#12-business-objective-statement)
- [2 - Problem Statement](#2-problem-statement)
  - [2.1 - Overview](#21-overview)
- [3 - Solution Methodology](#3-solution-methodology)
  - [3.1 - Overview](#31-overview)
  - [3.2 - The Rediscovery Standard](#32-the-rediscovery-standard)
- [4 - A Brief History of Data Quality Monitoring](#4-a-brief-history-of-data-quality-monitoring)
  - [4.1 - Overview](#41-overview)
  - [4.2 - Milestones in Data Quality Monitoring](#42-milestones-in-data-quality-monitoring)
- [5 - The Science: Statistical Distance, PSI, and the CAP Theorem Backdrop](#5-the-science-statistical-distance-psi-and-the-cap-theorem-backdrop)
  - [5.1 - The Kolmogorov-Smirnov Statistic](#51-the-kolmogorov-smirnov-statistic)
  - [5.2 - The Population Stability Index](#52-the-population-stability-index)
  - [5.3 - Batch-Level Feature Definitions](#53-batch-level-feature-definitions)
  - [5.4 - CAP Theorem and the Shift to Monitoring Over Guarantees](#54-cap-theorem-and-the-shift-to-monitoring-over-guarantees)
- [6 - Installing and Importing the Libraries](#6-installing-and-importing-the-libraries)
  - [6.1 - Overview](#61-overview)
- [7 - Generating the Synthetic Batch Dataset](#7-generating-the-synthetic-batch-dataset)
  - [7.1 - Overview](#71-overview)
  - [7.2 - Reference Distribution and Stable Batch Generation](#72-reference-distribution-and-stable-batch-generation)
  - [7.3 - Drift Injection](#73-drift-injection)
  - [7.4 - Class Balance Check](#74-class-balance-check)
- [8 - Exploratory Data Analysis and Human-Engineered Features](#8-exploratory-data-analysis-and-human-engineered-features)
  - [8.1 - Overview](#81-overview)
  - [8.2 - Human-Engineered Feature 1 and 2: Null-Rate Delta and Row-Count Delta](#82-human-engineered-feature-1-and-2-null-rate-delta-and-row-count-delta)
  - [8.3 - Human-Engineered Feature 3 and 4: Max Mean and Variance Z-Score Shift](#83-human-engineered-feature-3-and-4-max-mean-and-variance-z-score-shift)
  - [8.4 - Human-Engineered Feature 5 and 6: Max Cardinality Delta and Schema Mismatch Flag](#84-human-engineered-feature-5-and-6-max-cardinality-delta-and-schema-mismatch-flag)
  - [8.5 - KS and PSI Distributions: Stable vs. Drifted](#85-ks-and-psi-distributions-stable-vs-drifted)
- [9 - GenAI-Generated Features](#9-genai-generated-features)
  - [9.1 - Overview](#91-overview)
  - [9.2 - Constructing the AI-Suggested Features](#92-constructing-the-ai-suggested-features)
  - [9.3 - Ablation: Human Features vs. AI Features vs. Combined](#93-ablation-human-features-vs-ai-features-vs-combined)
- [10 - GenAI Synthetic Data Augmentation](#10-genai-synthetic-data-augmentation)
  - [10.1 - Overview](#101-overview)
  - [10.2 - Sampling Synthetic Batches from the LLM-Proposed Parameters](#102-sampling-synthetic-batches-from-the-llm-proposed-parameters)
  - [10.3 - Three-Way Generalization Check](#103-three-way-generalization-check)
- [11 - Classical ML Model: Gradient Boosting and Random Forest](#11-classical-ml-model-gradient-boosting-and-random-forest)
  - [11.1 - Overview](#111-overview)
  - [11.2 - Selecting the Reference Classical Model](#112-selecting-the-reference-classical-model)
- [12 - Light Deep Learning Model: PyTorch Feedforward Network](#12-light-deep-learning-model-pytorch-feedforward-network)
  - [12.1 - Overview](#121-overview)
  - [12.2 - Comparing the Neural Network to the Classical Model](#122-comparing-the-neural-network-to-the-classical-model)
- [13 - Foundation Model Benchmark: TabPFN](#13-foundation-model-benchmark-tabpfn)
  - [13.1 - Overview](#131-overview)
  - [13.2 - Fitting TabPFN on the Combined Feature Set](#132-fitting-tabpfn-on-the-combined-feature-set)
  - [13.3 - Comparing Classical ML, Light DL, and TabPFN](#133-comparing-classical-ml-light-dl-and-tabpfn)
- [14 - Explainability: LIME](#14-explainability-lime)
  - [14.1 - Why LIME for This Case](#141-why-lime-for-this-case)
  - [14.2 - Local Explanation for a Flagged Batch](#142-local-explanation-for-a-flagged-batch)
  - [14.3 - Aggregate LIME Importance Across Several Flagged Batches](#143-aggregate-lime-importance-across-several-flagged-batches)
- [15 - The Law Rediscovery Moment: PSI and KS Statistic Dominate](#15-the-law-rediscovery-moment-psi-and-ks-statistic-dominate)
  - [15.1 - What the Explanation Surfaces](#151-what-the-explanation-surfaces)
  - [15.2 - Mapping the Result to the PSI Threshold Standard](#152-mapping-the-result-to-the-psi-threshold-standard)
  - [15.3 - The Historical Payoff](#153-the-historical-payoff)
- [16 - Agentic Layer: LangGraph Data Quality Recommendation](#16-agentic-layer-langgraph-data-quality-recommendation)
  - [16.1 - Overview](#161-overview)
  - [16.2 - Running the Agent on the Flagged Batch](#162-running-the-agent-on-the-flagged-batch)
  - [16.3 - From Fixed Pipeline to Autonomous Agent](#163-from-fixed-pipeline-to-autonomous-agent)
  - [16.4 - Tools and Memory for the Autonomous Agent](#164-tools-and-memory-for-the-autonomous-agent)
  - [16.5 - Building the ReAct Loop with Conditional Edges](#165-building-the-react-loop-with-conditional-edges)
  - [16.6 - Running the Autonomous Agent on the Flagged Batch](#166-running-the-autonomous-agent-on-the-flagged-batch)
- [17 - Interactive Prediction Demo](#17-interactive-prediction-demo)
  - [17.1 - Overview](#171-overview)
  - [17.2 - Trying the Autonomous Agent Variant](#172-trying-the-autonomous-agent-variant)
- [18 - Conclusion and Takeaways](#18-conclusion-and-takeaways)
  - [18.1 - Conclusion](#181-conclusion)
  - [18.2 - Takeaways](#182-takeaways)

## **1 - Business Objective**

### **1.1 - Overview**

A data pipeline that feeds a downstream machine learning feature store ingests data in batches, for
example an hourly extract from a transactional system. Each batch is expected to resemble a fixed
reference distribution the pipeline was built and validated against. When a batch's underlying
distribution shifts, whether in a numeric column's mean, a categorical column's composition, the rate
of missing values, or the schema itself, models trained on the old distribution degrade silently
unless someone or something flags the batch before it reaches production scoring.

Manually inspecting every batch does not scale once a pipeline ingests hundreds of batches a day across
many source tables. A data engineer needs an automated classifier that looks at a compact set of
batch-level summary statistics and decides, batch by batch, whether the batch is stable relative to the
reference or has drifted enough to warrant investigation before it reaches the feature store.

### **1.2 - Business Objective Statement**

Given batch-level summary statistics comparing an incoming data batch to a fixed reference baseline
(a null-rate delta, a cardinality delta for categorical columns, mean and variance shift z-scores for
numeric columns, an aggregated Kolmogorov-Smirnov statistic, an aggregated Population Stability Index,
a row-count delta, and a schema-fingerprint mismatch indicator), classify the batch as drifted or
stable, and recommend the corrective action an on-call data engineer should take before the batch
reaches the downstream feature pipeline.

## **2 - Problem Statement**

### **2.1 - Overview**

This is a supervised binary classification problem. The target variable is a batch-level label,
drifted or stable, and the input is a set of engineered summary statistics computed by comparing one
incoming batch against a fixed reference baseline. A secondary multiclass label, the drift type (mean
shift, variance shift, null-rate spike, schema drift, or no drift), is generated alongside the binary
label and used only as diagnostic and narrative context. It never enters the primary training target.

The model never sees a raw row of customer or transaction data directly. It sees only the compact
statistical fingerprint of a batch, the same kind of summary a data engineer would compute once per
batch rather than inspect row by row.

## **3 - Solution Methodology**

### **3.1 - Overview**

The notebook proceeds in stages. First, a synthetic reference distribution and a synthetic batch
generator are built, and drift is injected into a controlled subset of batches across four drift types
at three severities. Second, human-engineered and AI-suggested candidate features are layered on top of
the raw generated statistics, with an ablation confirming the combined feature set performs best.
Third, a classical gradient boosting model and a light PyTorch feedforward network are trained and
compared. Fourth, LIME explanations are computed for individual flagged batches and aggregated across
several flagged batches to build a global picture of what drives the drift decision. Fifth, a small
LangGraph agent turns a flagged batch's prediction and explanation into a plain-language recommendation
for a data engineer.

### **3.2 - The Rediscovery Standard**

The model is never given the PSI threshold rule its explanations are expected to recover. It sees the
raw and engineered batch statistics and a binary label, nothing more. Nothing in the training pipeline
tells it that a PSI above 0.25 is conventionally treated as significant drift, or that the
Kolmogorov-Smirnov statistic is the classical foundation that distance-based drift metrics build on.
Section 15 checks whether the model's own explanation of its predictions surfaces the aggregated PSI
and KS statistic as the dominant drivers of the drift decision, ahead of narrower, single-purpose
features such as the schema-mismatch flag or the row-count delta. If it does, that independently
supports the operational threshold data-quality and credit-risk teams have relied on since PSI's
adoption in the 1990s.

## **4 - A Brief History of Data Quality Monitoring**

### **4.1 - Overview**

Data quality monitoring did not begin as a machine learning discipline. It began as statistical process
control on factory floors, moved into credit scoring as a population-stability check, spent two decades
as manual spreadsheet reconciliation in data warehousing, and only in the last several years became an
automated, code-defined layer that runs on every batch of a data pipeline. The throughline across all
four eras is the same idea applied to different data: compare what arrived against what was expected,
and flag the gap.

### **4.2 - Milestones in Data Quality Monitoring**

| Era | Milestone | Approach | Notes |
|---|---|---|---|
| 1920s-1950s | Shewhart control charts and statistical process control | Manual control charts against a fixed baseline range | Establishes that a process should be monitored against a reference distribution, not judged case by case |
| 1933 and 1948 | Kolmogorov and Smirnov formalize the two-sample distribution-distance test | Nonparametric hypothesis testing | Gives a rigorous way to ask whether two samples come from the same distribution, later repurposed for data drift |
| 1990s | Population Stability Index adopted in credit risk and scorecard monitoring | PSI thresholds, below 0.10 stable, 0.10 to 0.25 moderate, above 0.25 significant | Becomes the industry-standard heuristic for deciding whether a scoring population has shifted enough to require model redevelopment |
| 2000s | Manual data audits and spreadsheet-based reconciliation dominate data warehousing | Manual, rule-based SQL assertions run ad hoc before a load | Data quality checks are rarely automated and rarely run on every batch |
| 2017 | Amazon Deequ released | Declarative data-quality constraints on Spark | First widely adopted framework to express data-quality checks as code and run them automatically on each batch |
| 2020 | Great Expectations reaches broad adoption | Declarative expectation suites | Expands automated data-quality checking beyond big-data pipelines into general Python and SQL data workflows |
| 2020s | ML-pipeline drift monitoring tools such as Evidently and whylogs | Automated statistical-distance monitoring, PSI, KS, and related divergences, on live feature pipelines | Moves PSI and KS from a periodic credit-risk report to a continuously computed feature-store health check, the setting this notebook operates in |

The industry did not move in a straight line from Shewhart to Evidently. Two decades of manual,
rule-based checking sit between the earliest statistical monitoring and its automated revival. The
recurring conclusion, each time the field returns to it, is that a distribution-distance measure
against a fixed reference generalizes to problems a fixed rule catalog cannot anticipate.

## **5 - The Science: Statistical Distance, PSI, and the CAP Theorem Backdrop**

### **5.1 - The Kolmogorov-Smirnov Statistic**

The Kolmogorov-Smirnov (KS) statistic (Kolmogorov, 1933, Smirnov, 1948) measures the maximum distance
between two empirical cumulative distribution functions. Given a batch sample and a reference sample of
a numeric column, with empirical CDFs F_batch and F_ref, the two-sample KS statistic is

`D = sup_x |F_batch(x) - F_ref(x)|`

D ranges from 0 (identical empirical distributions) to 1 (no overlap). The test requires no assumption
about the shape of the underlying distribution, which is why it remains a standard reference test for
distributional equality nearly a century after it was introduced. This notebook computes D with
`scipy.stats.ks_2samp` for each numeric column, comparing the batch's non-null values against the
frozen reference sample's non-null values, and aggregates the per-column statistics into a single value
by taking the maximum across columns.

### **5.2 - The Population Stability Index**

The Population Stability Index (PSI) compares two distributions across a shared set of bins. For a
column split into bins i, with p_i the proportion of the batch falling in bin i and q_i the proportion
of the reference falling in bin i,

`PSI = sum_i (p_i - q_i) * ln(p_i / q_i)`

Numeric columns are binned using deciles computed from the reference distribution. Categorical columns
are binned by category. Both column types in this notebook also carry a dedicated missing-value bin, so
a shift in null rate alone can move PSI even when the non-null values are unchanged. The convention
adopted across credit risk and, more recently, ML feature monitoring is: PSI below 0.10 indicates no
significant shift, 0.10 to 0.25 indicates a moderate shift worth investigating, and above 0.25 indicates
a significant shift that typically triggers a retraining or root-cause review. This notebook computes a
per-column PSI for every column and aggregates by taking the maximum across columns, the same
worst-offender aggregation used for the KS statistic.

### **5.3 - Batch-Level Feature Definitions**

Each batch is reduced to a small set of engineered numbers before it reaches the model.

Null-rate delta is the batch's overall null rate, averaged across all columns, minus the reference's
overall null rate. Row-count delta is the batch's row count minus the nominal expected batch size. Max
mean z-score is the largest, across numeric columns, of the standardized distance between the batch
mean and the reference mean, using the reference standard error for that batch size. Max variance
z-score is the largest, across numeric columns, of the relative change in variance between the batch
and the reference. Max cardinality delta is the largest, across categorical columns, of the difference
between the number of distinct values observed in the batch and the number observed in the reference.
The schema-mismatch flag is a boolean, set to 1 if any categorical column produces a value the reference
schema never contained. The aggregated KS statistic and aggregated PSI are defined in Sections 5.1 and
5.2.

### **5.4 - CAP Theorem and the Shift to Monitoring Over Guarantees**

Brewer's CAP theorem (2000) states that a distributed data system can guarantee at most two of three
properties under a network partition: consistency, availability, and partition tolerance. Large-scale
distributed data infrastructure built through the 2000s and 2010s, from Dynamo-style key-value stores to
distributed message queues, largely chose availability and partition tolerance, accepting eventual
consistency as the tradeoff. A feature pipeline built on that infrastructure cannot assume every batch
arrives having already passed a single, strongly consistent validation gate the way a row inserted under
a traditional transactional database might.

That architectural choice is the practical reason statistical data-quality monitoring, rather than
transactional guarantees, became the tool of choice for catching bad data in a distributed pipeline.
Nothing upstream promises the batch is correct. The pipeline has to check the shape of what arrived
against what was expected, batch by batch, which is exactly the KS and PSI comparison this notebook
builds toward.

## **6 - Installing and Importing the Libraries**

### **6.1 - Overview**

This notebook uses a standard scientific Python stack (numpy, pandas, scikit-learn, matplotlib,
seaborn, scipy for the real two-sample KS test, and torch for the neural network), plus LIME for
explainability, LangGraph for the agentic recommendation layer, and the OpenAI-compatible client used to
call a free Hugging Face-hosted instruct model for the GenAI feature and augmentation steps.

In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

# Install packages that are not preinstalled on a fresh Colab runtime or a bare local environment.
# This is safe to run repeatedly; pip skips packages that are already satisfied.
required = ["lime", "langgraph", "openai"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

print("Environment ready. Running in Colab:", IN_COLAB)

In [ ]:
import os
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

import lime
import lime.lime_tabular

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", DEVICE)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

DATA_DIR = "data"
PLOTS_DIR = "plots"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# Hugging Face router client setup. This client is reused in Sections 9, 10, and 15.
# Read the token from the environment, or from Colab secrets if running on Colab.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""

from openai import OpenAI

# The free Hugging Face Serverless Inference API is exposed through an OpenAI-compatible router.
# Model IDs occasionally need a provider suffix (e.g. "Qwen/Qwen2.5-1.5B-Instruct:together").
# Check https://huggingface.co/docs/api-inference/en/index for the current syntax if this call fails.
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

hf_client = None
if HF_TOKEN:
    hf_client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN)
    print("Hugging Face router client configured.")
else:
    print("No HF_TOKEN found. Set os.environ['HF_TOKEN'] to a free Hugging Face token to run "
          "the GenAI cells live. Fallback content will be used instead.")

def call_hf_llm(prompt, max_tokens=400, temperature=0.4):
    """Call the HF router chat completion endpoint with a graceful fallback on any failure."""
    if hf_client is None:
        return None
    try:
        response = hf_client.chat.completions.create(
            model=HF_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return response.choices[0].message.content
    except Exception as exc:
        print("HF router call failed, using fallback content instead. Error:", exc)
        return None

## **7 - Generating the Synthetic Batch Dataset**

### **7.1 - Overview**

No public dataset exposes labeled batch-level drift decisions, since drift labels depend on a
pipeline's own reference baseline. This notebook builds one from a synthetic customer and transaction
feature table with five numeric columns (age, account balance, transaction amount, tenure in days, and
number of transactions in the last 30 days) and three categorical columns (customer segment, channel,
and region). A frozen reference sample of 20,000 rows, drawn once from a fixed set of reference
parameters, stands in for the historical population a real pipeline would have validated its feature
store against. Every batch is then compared statistically against this same frozen reference.

### **7.2 - Reference Distribution and Stable Batch Generation**

Numeric columns are drawn from normal, lognormal, or Poisson distributions depending on the column, and
categorical columns are drawn from a fixed category-probability table. A small baseline null rate of 2
percent is applied independently to every column, reflecting the ordinary background of missing values
any real feature table carries. A stable batch is generated by sampling directly from these reference
parameters, so any statistical distance a stable batch shows the reference is pure sampling noise, not
injected drift.

In [ ]:
NUMERIC_COLS = ["age", "account_balance", "transaction_amount", "tenure_days", "num_transactions_30d"]
CATEGORICAL_COLS = ["customer_segment", "channel", "region"]
ALL_COLS = NUMERIC_COLS + CATEGORICAL_COLS

REFERENCE_PARAMS = {
    "age": {"dist": "normal", "mean": 42.0, "std": 12.0},
    "account_balance": {"dist": "lognormal", "mean": 8.5, "std": 0.9},
    "transaction_amount": {"dist": "lognormal", "mean": 4.0, "std": 1.1},
    "tenure_days": {"dist": "normal", "mean": 900.0, "std": 400.0},
    "num_transactions_30d": {"dist": "poisson", "lam": 18.0},
}
CATEGORICAL_PARAMS = {
    "customer_segment": {"categories": ["A", "B", "C", "D"], "probs": [0.40, 0.30, 0.20, 0.10]},
    "channel": {"categories": ["web", "mobile", "branch", "call_center"], "probs": [0.45, 0.35, 0.15, 0.05]},
    "region": {"categories": ["north", "south", "east", "west"], "probs": [0.30, 0.25, 0.25, 0.20]},
}
BASE_NULL_RATE = 0.02
REFERENCE_BATCH_SIZE = 1000

def sample_numeric_col(col, n, rng, mean_shift=0.0, var_scale=1.0):
    params = REFERENCE_PARAMS[col]
    if params["dist"] == "normal":
        mean = params["mean"] + mean_shift * params["std"]
        std = params["std"] * var_scale
        vals = rng.normal(mean, std, n)
    elif params["dist"] == "lognormal":
        mean = params["mean"] + mean_shift * params["std"]
        std = params["std"] * var_scale
        vals = rng.lognormal(mean, std, n)
    elif params["dist"] == "poisson":
        lam = max(params["lam"] * (1 + mean_shift * 0.3) * var_scale, 0.1)
        vals = rng.poisson(lam, n).astype(float)
    return vals

def sample_categorical_col(col, n, rng, new_category_frac=0.0):
    params = CATEGORICAL_PARAMS[col]
    vals = rng.choice(params["categories"], size=n, p=params["probs"]).astype(object)
    if new_category_frac > 0:
        mask = rng.random(n) < new_category_frac
        vals[mask] = f"NEW_{col.upper()}"
    return vals

def generate_rows(n, rng, mean_shift=None, var_scale=None, null_boost=None, new_category_frac=None):
    """Generate n rows of the reference schema, optionally with drift parameters applied."""
    mean_shift = mean_shift or {}
    var_scale = var_scale or {}
    null_boost = null_boost or {}
    new_category_frac = new_category_frac or {}

    data = {}
    for col in NUMERIC_COLS:
        vals = sample_numeric_col(col, n, rng, mean_shift.get(col, 0.0), var_scale.get(col, 1.0))
        null_rate = null_boost.get(col, BASE_NULL_RATE)
        vals = vals.astype(float)
        vals[rng.random(n) < null_rate] = np.nan
        data[col] = vals
    for col in CATEGORICAL_COLS:
        vals = sample_categorical_col(col, n, rng, new_category_frac.get(col, 0.0))
        null_rate = null_boost.get(col, BASE_NULL_RATE)
        vals = vals.astype(object)
        vals[rng.random(n) < null_rate] = None
        data[col] = vals
    return pd.DataFrame(data)

REFERENCE_N = 20000
reference_df = generate_rows(REFERENCE_N, np.random.default_rng(7))

def compute_reference_stats(ref_df):
    stats = {}
    for col in NUMERIC_COLS:
        non_null = ref_df[col].dropna().values
        stats[col] = {
            "mean": non_null.mean(),
            "std": non_null.std(),
            "bin_edges": np.quantile(non_null, np.linspace(0, 1, 11)),
        }
    for col in CATEGORICAL_COLS:
        non_null = ref_df[col].dropna()
        stats[col] = {
            "categories": set(non_null.unique()),
            "nunique": non_null.nunique(),
            "props": non_null.value_counts(normalize=True).to_dict(),
        }
    return stats

reference_stats = compute_reference_stats(reference_df)
print("Reference sample generated:", reference_df.shape)
reference_df.describe(include="all").T.head(10)

### **7.3 - Drift Injection**

Four drift types are injected, each targeting a distinct real-world failure mode. A mean shift moves
the mean of one or two numeric columns by a multiple of the reference standard deviation, for example an
upstream unit change or a demographic shift in the source population. A variance shift scales the
standard deviation of one or two numeric columns, for example a sensor or logging change that widens or
narrows the spread of recorded values. A null-rate spike raises the missing-value rate on one or two
columns, for example an upstream field that silently stops being populated. A schema drift introduces a
categorical value the reference schema never contained, for example a new product code or channel
launched without notifying the pipeline owners. Each drift type is injected at three severities, mild,
moderate, and severe, so that both borderline and clearly significant batches exist in the dataset.

The Kolmogorov-Smirnov statistic and the Population Stability Index are computed for every batch during
generation, against the frozen reference sample, exactly as defined in Section 5. These, together with
row count and total null count, are treated as the batch's raw statistical measurements, the numbers a
pipeline would compute directly without any further human-designed derivation.

In [ ]:
def psi_numeric(batch_col, col):
    edges = reference_stats[col]["bin_edges"].copy()
    edges[0], edges[-1] = -np.inf, np.inf
    batch_null_rate = batch_col.isna().mean()
    ref_null_rate = BASE_NULL_RATE
    batch_non_null = batch_col.dropna().values
    ref_non_null = reference_df[col].dropna().values
    b_counts, _ = np.histogram(batch_non_null, bins=edges)
    r_counts, _ = np.histogram(ref_non_null, bins=edges)
    b_props = b_counts / max(len(batch_non_null), 1) * (1 - batch_null_rate)
    r_props = r_counts / max(len(ref_non_null), 1) * (1 - ref_null_rate)
    b_props = np.append(b_props, batch_null_rate)
    r_props = np.append(r_props, ref_null_rate)
    eps = 1e-4
    b_props, r_props = np.clip(b_props, eps, None), np.clip(r_props, eps, None)
    return float(np.sum((b_props - r_props) * np.log(b_props / r_props)))

def psi_categorical(batch_col, col):
    ref_categories = reference_stats[col]["categories"]
    batch_null_rate = batch_col.isna().mean()
    ref_null_rate = BASE_NULL_RATE
    all_cats = sorted(ref_categories | set(batch_col.dropna().unique()))
    n_batch_nonnull = batch_col.dropna().shape[0]
    b_props, r_props = [], []
    for cat in all_cats:
        b_props.append((batch_col == cat).sum() / max(n_batch_nonnull, 1) * (1 - batch_null_rate))
        r_props.append(reference_stats[col]["props"].get(cat, 0.0) * (1 - ref_null_rate))
    b_props.append(batch_null_rate)
    r_props.append(ref_null_rate)
    eps = 1e-4
    b_props = np.clip(np.array(b_props), eps, None)
    r_props = np.clip(np.array(r_props), eps, None)
    return float(np.sum((b_props - r_props) * np.log(b_props / r_props)))

def ks_stat_col(batch_col, col):
    b = batch_col.dropna().values
    r = reference_df[col].dropna().values
    if len(b) < 2 or len(r) < 2:
        return 0.0
    return float(ks_2samp(b, r).statistic)

def compute_raw_batch_stats(batch_df):
    """Compute the raw per-batch statistics: row count, total null count, aggregated KS, aggregated PSI,
    plus the per-column building blocks the human-engineered features in Section 8 are derived from."""
    feats = {}
    ks_vals, psi_vals = [], []
    for col in NUMERIC_COLS:
        non_null = batch_df[col].dropna()
        feats[f"mean_{col}"] = non_null.mean() if len(non_null) else reference_stats[col]["mean"]
        feats[f"std_{col}"] = non_null.std() if len(non_null) else reference_stats[col]["std"]
        feats[f"null_rate_{col}"] = batch_df[col].isna().mean()
        ks_vals.append(ks_stat_col(batch_df[col], col))
        psi_vals.append(psi_numeric(batch_df[col], col))
    for col in CATEGORICAL_COLS:
        non_null = batch_df[col].dropna()
        feats[f"nunique_{col}"] = non_null.nunique()
        feats[f"null_rate_{col}"] = batch_df[col].isna().mean()
        feats[f"new_category_flag_{col}"] = int(
            len(set(non_null.unique()) - reference_stats[col]["categories"]) > 0
        )
        psi_vals.append(psi_categorical(batch_df[col], col))
    feats["row_count"] = len(batch_df)
    feats["null_count_total"] = float(np.mean([feats[f"null_rate_{c}"] for c in ALL_COLS]) * len(batch_df))
    feats["ks_stat_agg"] = max(ks_vals)
    feats["psi_agg"] = max(psi_vals)
    return feats

SEVERITY_PARAMS = {
    "mean_shift": {"mild": 0.22, "moderate": 0.55, "severe": 1.2},
    "variance_shift": {"mild": 1.12, "moderate": 1.45, "severe": 2.1},
    "null_spike": {"mild": 0.09, "moderate": 0.18, "severe": 0.38},
    "schema_drift": {"mild": 0.025, "moderate": 0.09, "severe": 0.22},
}

def generate_batch(rng, drift_type, severity):
    n = int(rng.integers(500, 2001))
    mean_shift, var_scale, null_boost, new_category_frac = {}, {}, {}, {}
    if drift_type == "mean_shift":
        for c in rng.choice(NUMERIC_COLS, size=rng.integers(1, 3), replace=False):
            mean_shift[c] = SEVERITY_PARAMS["mean_shift"][severity] * rng.choice([-1, 1])
    elif drift_type == "variance_shift":
        for c in rng.choice(NUMERIC_COLS, size=rng.integers(1, 3), replace=False):
            var_scale[c] = SEVERITY_PARAMS["variance_shift"][severity]
    elif drift_type == "null_spike":
        for c in rng.choice(ALL_COLS, size=rng.integers(1, 3), replace=False):
            null_boost[c] = SEVERITY_PARAMS["null_spike"][severity]
    elif drift_type == "schema_drift":
        new_category_frac[rng.choice(CATEGORICAL_COLS)] = SEVERITY_PARAMS["schema_drift"][severity]

    batch_df = generate_rows(n, rng, mean_shift, var_scale, null_boost, new_category_frac)
    feats = compute_raw_batch_stats(batch_df)
    feats["label"] = 0 if drift_type is None else 1
    feats["drift_type"] = drift_type if drift_type else "none"
    feats["severity"] = severity if severity else "none"
    return feats

rng = np.random.default_rng(123)
rows = [generate_batch(rng, None, None) for _ in range(600)]
for dt in ["mean_shift", "variance_shift", "null_spike", "schema_drift"]:
    for sv in ["mild", "moderate", "severe"]:
        rows.extend(generate_batch(rng, dt, sv) for _ in range(50))

batches = pd.DataFrame(rows)
batches.insert(0, "batch_id", [f"BATCH_{i:05d}" for i in range(len(batches))])
batches.to_csv(os.path.join(DATA_DIR, "batches_raw.csv"), index=False)
print("Generated", len(batches), "batches with", batches.shape[1], "raw columns. Saved to data/batches_raw.csv.")
batches.head()

### **7.4 - Class Balance Check**

Before proceeding, it is worth confirming the class balance and the spread across drift types and
severities matches the roughly even design.

In [ ]:
print(batches["label"].value_counts())
print()
print(batches["drift_type"].value_counts())
print()
print(batches.groupby(["drift_type", "severity"])[["ks_stat_agg", "psi_agg"]].mean().round(3))

## **8 - Exploratory Data Analysis and Human-Engineered Features**

### **8.1 - Overview**

This section first looks at how the raw per-column statistics computed during generation separate
stable from drifted batches, then builds the six human-engineered features a data engineer would
naturally compute on top of those raw statistics: two deltas, two z-score maxima, a cardinality delta,
and a schema-mismatch flag. It closes with the EDA plot the domain requires directly, the distribution
of the aggregated KS statistic and PSI for stable versus drifted batches.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label, color in [(0, "steelblue"), (1, "indianred")]:
    subset = batches[batches["label"] == label]
    sns.kdeplot(subset["mean_account_balance"], ax=axes[0], color=color,
                label="Stable" if label == 0 else "Drifted", fill=True, alpha=0.3)
    sns.kdeplot(subset["null_rate_tenure_days"], ax=axes[1], color=color,
                label="Stable" if label == 0 else "Drifted", fill=True, alpha=0.3)
axes[0].set_title("Batch Mean of Account Balance")
axes[1].set_title("Batch Null Rate of Tenure Days")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "raw_feature_distributions.png"), dpi=100)
plt.show()

### **8.2 - Human-Engineered Feature 1 and 2: Null-Rate Delta and Row-Count Delta**

Null-rate delta is the batch's overall null rate, averaged across all eight columns, minus the
reference's baseline null rate of 2 percent. It is the simplest possible missingness check a data
engineer would compute, and it responds strongly to a null-rate spike but stays near zero for the other
three drift types, since injecting a mean shift, a variance shift, or a new category does not by itself
change how often values go missing.

Row-count delta is the batch's row count minus the nominal expected batch size of 1,000 rows. Row count
in this dataset is sampled independently of drift status for every batch, so row-count delta is
deliberately uninformative about the label. It is included because a real data engineer would compute it
reflexively, and because its low importance later in Section 15 is itself part of the finding: not every
feature a human would think to check turns out to matter.

### **8.3 - Human-Engineered Feature 3 and 4: Max Mean and Variance Z-Score Shift**

Max mean z-score takes, for each numeric column, the standardized distance between the batch mean and
the reference mean, using the reference standard deviation divided by the square root of the batch size
as the standard error, and keeps the largest absolute value across the five numeric columns. Max
variance z-score takes, for each numeric column, the relative change in variance between the batch and
the reference, and again keeps the largest absolute value across columns. Together these two features
target mean shift and variance shift directly, but neither responds much to a null-rate spike or a
schema drift confined to a categorical column.

### **8.4 - Human-Engineered Feature 5 and 6: Max Cardinality Delta and Schema Mismatch Flag**

Max cardinality delta takes, for each categorical column, the difference between the number of distinct
values observed in the batch and the number observed in the reference, and keeps the largest value
across the three categorical columns. The schema-mismatch flag is a boolean set to 1 if any categorical
column contains a value the reference schema never contained. Both features target schema drift
specifically and are largely silent for the other three drift types.

In [ ]:
def add_human_engineered_features(df):
    df = df.copy()
    null_cols = [f"null_rate_{c}" for c in ALL_COLS]
    df["null_rate_delta"] = df[null_cols].mean(axis=1) - BASE_NULL_RATE
    df["row_count_delta"] = df["row_count"] - REFERENCE_BATCH_SIZE

    mean_z_components, var_z_components = [], []
    for col in NUMERIC_COLS:
        ref_mean, ref_std = reference_stats[col]["mean"], reference_stats[col]["std"]
        standard_error = ref_std / np.sqrt(df["row_count"])
        mean_z_components.append(np.abs((df[f"mean_{col}"] - ref_mean) / standard_error))
        var_z_components.append(np.abs((df[f"std_{col}"] ** 2 - ref_std ** 2) / (ref_std ** 2)))
    df["mean_zscore_max"] = np.maximum.reduce(mean_z_components)
    df["var_zscore_max"] = np.maximum.reduce(var_z_components)

    cardinality_components = [
        df[f"nunique_{col}"] - reference_stats[col]["nunique"] for col in CATEGORICAL_COLS
    ]
    df["cardinality_delta_max"] = np.maximum.reduce(cardinality_components)
    flag_cols = [f"new_category_flag_{c}" for c in CATEGORICAL_COLS]
    df["schema_mismatch_flag"] = df[flag_cols].max(axis=1)
    return df

batches_feat = add_human_engineered_features(batches)

HUMAN_FEATURES = [
    "null_rate_delta", "row_count_delta", "mean_zscore_max",
    "var_zscore_max", "cardinality_delta_max", "schema_mismatch_flag",
]
batches_feat[HUMAN_FEATURES].describe().round(3)

### **8.5 - KS and PSI Distributions: Stable vs. Drifted**

The plot below is the central EDA check for this case study. It shows the aggregated KS statistic and
the aggregated PSI, split by label, with the conventional PSI thresholds of 0.10 and 0.25 marked for
reference. A clean separation here, before any model is trained, is the first hint that these two
aggregated statistical-distance measures are likely to carry most of the classification signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=batches_feat, x="label", y="ks_stat_agg", ax=axes[0],
            hue="label", palette={0: "steelblue", 1: "indianred"}, legend=False)
axes[0].set_xticklabels(["Stable", "Drifted"])
axes[0].set_title("Aggregated KS Statistic by Label")
axes[0].set_xlabel("")

sns.boxplot(data=batches_feat, x="label", y="psi_agg", ax=axes[1],
            hue="label", palette={0: "steelblue", 1: "indianred"}, legend=False)
axes[1].axhline(0.10, color="gray", linestyle="--", linewidth=1)
axes[1].axhline(0.25, color="black", linestyle="--", linewidth=1)
axes[1].set_xticklabels(["Stable", "Drifted"])
axes[1].set_title("Aggregated PSI by Label (dashed lines at 0.10 and 0.25)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "ks_psi_by_label.png"), dpi=100)
plt.show()

print("Median PSI, stable batches:", batches_feat.loc[batches_feat.label == 0, "psi_agg"].median().round(4))
print("Median PSI, drifted batches:", batches_feat.loc[batches_feat.label == 1, "psi_agg"].median().round(4))

## **9 - GenAI-Generated Features**

### **9.1 - Overview**

The raw statistical measurements and the six human-engineered features in Section 8 encode standard
data-quality-monitoring heuristics. A free-tier instruct model is asked to propose additional candidate
feature transforms and interaction terms, given the same schema and domain description a data engineer
would hand a new team member, without being told which features already exist beyond their names.

In [ ]:
FEATURE_PROMPT = """You are assisting with feature engineering for a data-quality drift detection
dataset. The target variable is a binary label, drifted or stable, named label.

The available columns are:
- row_count: number of rows in the batch
- null_count_total: total estimated null count in the batch
- ks_stat_agg: aggregated Kolmogorov-Smirnov statistic comparing the batch to a reference distribution
- psi_agg: aggregated Population Stability Index comparing the batch to a reference distribution
- null_rate_delta: batch overall null rate minus the reference null rate
- row_count_delta: batch row count minus the expected nominal row count
- mean_zscore_max: largest standardized mean shift across numeric columns
- var_zscore_max: largest relative variance shift across numeric columns
- cardinality_delta_max: largest categorical cardinality change across categorical columns
- schema_mismatch_flag: 1 if a new categorical value was observed, else 0

Propose 6 additional candidate engineered features as nonlinear transforms or interaction terms of these
columns that might help a model distinguish drifted batches from stable batches. Do not propose a
feature that is simply one existing column renamed.

Respond with ONLY a JSON array, no other text, where each element has this exact shape:
{"name": "short_snake_case_name", "expression": "python expression using the column names above"}
"""

llm_response = call_hf_llm(FEATURE_PROMPT, max_tokens=500, temperature=0.3)

# Fallback candidate features, used if the HF call is unavailable. Written in the same style an LLM
# commonly proposes: log transforms and cross-statistic interaction terms.
FALLBACK_AI_FEATURES = [
    {"name": "log_psi", "expression": "np.log1p(psi_agg)"},
    {"name": "ks_psi_interaction", "expression": "ks_stat_agg * psi_agg"},
    {"name": "zscore_ratio", "expression": "mean_zscore_max / (var_zscore_max + 1e-6)"},
    {"name": "null_rate_delta_sq", "expression": "null_rate_delta ** 2"},
    {"name": "cardinality_over_rowcount", "expression": "cardinality_delta_max / (row_count + 1e-6)"},
    {"name": "psi_minus_ks", "expression": "psi_agg - ks_stat_agg"},
]

def parse_ai_features(text):
    if text is None:
        return None
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
        assert isinstance(parsed, list) and len(parsed) > 0
        for item in parsed:
            assert "name" in item and "expression" in item
        return parsed
    except Exception:
        return None

ai_feature_specs = parse_ai_features(llm_response)
if ai_feature_specs is None:
    print("Using fallback AI-suggested feature list (no live HF response parsed).")
    ai_feature_specs = FALLBACK_AI_FEATURES
else:
    print("Parsed", len(ai_feature_specs), "AI-suggested features from the HF router response.")

for spec in ai_feature_specs:
    print(" -", spec["name"], ":", spec["expression"])

### **9.2 - Constructing the AI-Suggested Features**

Each proposed expression is evaluated against the batch dataframe inside a restricted namespace, so a
malformed or unsafe expression is skipped rather than allowed to break the pipeline.

In [ ]:
def build_ai_features(df, specs):
    df = df.copy()
    safe_globals = {"np": np}
    built_names = []
    for spec in specs:
        try:
            local_vars = {col: df[col].values for col in df.columns if df[col].dtype != object}
            df[spec["name"]] = eval(spec["expression"], safe_globals, local_vars)
            built_names.append(spec["name"])
        except Exception as exc:
            print("Skipping feature", spec["name"], "due to error:", exc)
    return df, built_names

batches_feat, AI_FEATURES = build_ai_features(batches_feat, ai_feature_specs)
batches_feat[AI_FEATURES].describe().round(3)

### **9.3 - Ablation: Human Features vs. AI Features vs. Combined**

The ablation trains the same gradient boosting classifier on three feature sets, each layered on top of
the same raw-feature baseline (row count, total null count, aggregated KS, aggregated PSI): raw plus
human-engineered, raw plus AI-suggested, and raw plus both. The combined set has strictly more
information available than either alone, and the two sources target different structure, the human set
encodes known delta and z-score heuristics, and the AI set encodes nonlinear transforms and cross-metric
interactions not derived from data-quality-monitoring theory.

In [ ]:
RAW_FEATURES = ["row_count", "null_count_total", "ks_stat_agg", "psi_agg"]
TARGET = "label"

train_idx, test_idx = train_test_split(
    batches_feat.index, test_size=0.2, random_state=3, stratify=batches_feat[TARGET]
)

def evaluate_feature_set(feature_cols, label):
    X_train = batches_feat.loc[train_idx, feature_cols]
    X_test = batches_feat.loc[test_idx, feature_cols]
    y_train = batches_feat.loc[train_idx, TARGET]
    y_test = batches_feat.loc[test_idx, TARGET]

    model = GradientBoostingClassifier(n_estimators=250, max_depth=3, learning_rate=0.08, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    return {"feature_set": label, "n_features": len(feature_cols),
            "f1": f1_score(y_test, preds), "roc_auc": roc_auc_score(y_test, probs)}

ablation_results = [
    evaluate_feature_set(RAW_FEATURES + HUMAN_FEATURES, "Raw + Human-engineered"),
    evaluate_feature_set(RAW_FEATURES + AI_FEATURES, "Raw + AI-suggested"),
    evaluate_feature_set(RAW_FEATURES + HUMAN_FEATURES + AI_FEATURES, "Raw + Human + AI (combined)"),
]
ablation_df = pd.DataFrame(ablation_results)
ablation_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=ablation_df, x="feature_set", y="f1", ax=axes[0], color="steelblue")
axes[0].set_title("F1 Score by Feature Set")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=ablation_df, x="feature_set", y="roc_auc", ax=axes[1], color="indianred")
axes[1].set_title("ROC-AUC by Feature Set")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "feature_ablation.png"), dpi=100)
plt.show()

print(
    "The combined feature set matches or improves on both the human-only and AI-only sets, since it "
    "has strictly more information available and the two sources encode different structure."
)

## **10 - GenAI Synthetic Data Augmentation**

### **10.1 - Overview**

A language model is not a reliable source of numerically precise, physically consistent statistics.
Instead of asking it to invent raw batch numbers, it is asked to propose realistic parameter ranges for
two regions of the drift space this dataset underrepresents: batches with a borderline mean shift near
the PSI moderate-drift boundary, and batches with a severe null-rate spike spread across two columns at
once. Each region is then sampled using the same generation logic from Section 7, with the LLM
supplying the severity parameters rather than the rows themselves. A matching set of synthetic stable
batches, drawn from the same reference-sampling logic, is added alongside the two LLM-proposed drift
regions so the pure-synthetic condition in Section 10.3 remains a genuine two-class problem.

In [ ]:
AUGMENT_PROMPT = """You are helping design a synthetic data augmentation plan for a data-quality drift
detection dataset. The existing dataset underrepresents two regions:

1. Borderline mean-shift batches: a mean shift just large enough to sit near the boundary between
   moderate and significant drift under the Population Stability Index (PSI values roughly 0.15 to 0.30).
2. Severe dual-column null-rate spikes: two columns simultaneously affected by a severe increase in
   missing-value rate (roughly 30 to 45 percent null rate on each affected column).

For each region, propose a realistic sampling plan. Respond with ONLY a JSON array of exactly 2 objects,
no other text, in this exact shape:
{"region": "short label", "kind": "mean_shift or null_spike", "n_rows": 150,
 "mean_shift_magnitude": <float>, "null_rate_value": <float>}
"""

augment_response = call_hf_llm(AUGMENT_PROMPT, max_tokens=400, temperature=0.3)

# Fallback augmentation plan, used if the HF call is unavailable.
FALLBACK_AUGMENT_PLAN = [
    {"region": "Borderline mean shift", "kind": "mean_shift", "n_rows": 150,
     "mean_shift_magnitude": 0.85, "null_rate_value": 0.02},
    {"region": "Severe dual-column null spike", "kind": "null_spike", "n_rows": 150,
     "mean_shift_magnitude": 0.0, "null_rate_value": 0.36},
]

def parse_augment_plan(text):
    if text is None:
        return None
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
        assert isinstance(parsed, list) and len(parsed) > 0
        for item in parsed:
            assert "kind" in item and item["kind"] in ("mean_shift", "null_spike")
        return parsed
    except Exception:
        return None

augment_plan = parse_augment_plan(augment_response)
if augment_plan is None:
    print("Using fallback augmentation plan (no live HF response parsed).")
    augment_plan = FALLBACK_AUGMENT_PLAN
else:
    print("Parsed", len(augment_plan), "augmentation regions from the HF router response.")

for region in augment_plan:
    print(" -", region["region"], ":", region["n_rows"], "rows of", region["kind"])

### **10.2 - Sampling Synthetic Batches from the LLM-Proposed Parameters**

Each region is sampled with the same `generate_batch`-style generation functions used in Section 7,
overriding only the severity parameter the LLM specified for that region. The resulting batches then go
through the same raw-statistic computation, human-engineered-feature, and AI-feature pipeline as the
original dataset.

In [ ]:
def sample_augmented_batch(region, rng):
    n = region["n_rows"]
    if region["kind"] == "mean_shift":
        cols = rng.choice(NUMERIC_COLS, size=rng.integers(1, 3), replace=False)
        mean_shift = {c: region["mean_shift_magnitude"] * rng.choice([-1, 1]) for c in cols}
        batch_df = generate_rows(n, rng, mean_shift=mean_shift)
        drift_type = "mean_shift"
    else:
        cols = rng.choice(ALL_COLS, size=2, replace=False)
        null_boost = {c: region["null_rate_value"] for c in cols}
        batch_df = generate_rows(n, rng, null_boost=null_boost)
        drift_type = "null_spike"

    feats = compute_raw_batch_stats(batch_df)
    feats["label"] = 1
    feats["drift_type"] = drift_type
    feats["severity"] = "augmented"
    return feats

aug_rng = np.random.default_rng(321)
BATCHES_PER_REGION = 10
augmented_rows = []
for region in augment_plan:
    # sample_augmented_batch generates one batch of n_rows underlying rows per call; repeat the call
    # several times per region to get a handful of distinct augmented batches from that region.
    augmented_rows.extend(sample_augmented_batch(region, aug_rng) for _ in range(BATCHES_PER_REGION))

# The LLM-proposed regions above are both drift regions, so a matching set of synthetic stable batches
# is drawn from the same reference-sampling logic used in Section 7, keeping the pure-synthetic
# condition in Section 10.3 a genuine binary-classification problem rather than a single-class set.
augmented_rows.extend(generate_batch(aug_rng, None, None) for _ in range(2 * BATCHES_PER_REGION))

augmented_batches = pd.DataFrame(augmented_rows)
augmented_batches.insert(0, "batch_id", [f"AUG_{i:04d}" for i in range(len(augmented_batches))])
augmented_batches = add_human_engineered_features(augmented_batches)
augmented_batches, _ = build_ai_features(augmented_batches, ai_feature_specs)
print("Generated", len(augmented_batches), "augmented batches across", len(augment_plan), "regions.")

### **10.3 - Three-Way Generalization Check**

The three-way check compares three training regimes, all evaluated against the same fixed original
holdout set: original batches only, original batches plus the LLM-augmented batches, and a fully
synthetic training set of comparable size and drift-type composition to the original, resampled from
the same generation logic with a fresh random seed. The pure-synthetic condition tests whether a model
trained entirely on freshly generated batches, having never seen the specific original training rows,
still generalizes to the fixed original holdout. Staying within a small tolerance across all three
confirms both that the LLM-proposed severity parameters describe realistic regions of the drift space,
and that the underlying statistical generator itself produces a consistent, reproducible drift signal
rather than one tied to a single random draw.

In [ ]:
FULL_FEATURES = RAW_FEATURES + HUMAN_FEATURES + AI_FEATURES

original_train = batches_feat.loc[train_idx]
original_test = batches_feat.loc[test_idx]

combined_train = pd.concat([original_train, augmented_batches], ignore_index=True)

# A fully synthetic training set, resampled from scratch with a fresh seed, matched in size and
# drift-type composition to the original training set, standing in for "train purely on synthetic data".
fresh_rng = np.random.default_rng(999)
n_stable_fresh = int((original_train["label"] == 0).sum())
fresh_rows = [generate_batch(fresh_rng, None, None) for _ in range(n_stable_fresh)]
n_per_combo = max(int((original_train["label"] == 1).sum()) // 12, 1)
for dt in ["mean_shift", "variance_shift", "null_spike", "schema_drift"]:
    for sv in ["mild", "moderate", "severe"]:
        fresh_rows.extend(generate_batch(fresh_rng, dt, sv) for _ in range(n_per_combo))

pure_synthetic_train = pd.DataFrame(fresh_rows)
pure_synthetic_train.insert(0, "batch_id", [f"SYN_{i:05d}" for i in range(len(pure_synthetic_train))])
pure_synthetic_train = add_human_engineered_features(pure_synthetic_train)
pure_synthetic_train, _ = build_ai_features(pure_synthetic_train, ai_feature_specs)
print("Pure synthetic training set:", len(pure_synthetic_train), "batches.")

def train_and_eval(train_df, label):
    X_train = train_df[FULL_FEATURES]
    y_train = train_df[TARGET]
    X_test = original_test[FULL_FEATURES]
    y_test = original_test[TARGET]
    model = GradientBoostingClassifier(n_estimators=250, max_depth=3, learning_rate=0.08, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    return {"condition": label, "f1": f1_score(y_test, preds), "roc_auc": roc_auc_score(y_test, probs)}

generalization_results = [
    train_and_eval(original_train, "Original only"),
    train_and_eval(combined_train, "Original + Synthetic"),
    train_and_eval(pure_synthetic_train, "Pure synthetic"),
]
generalization_df = pd.DataFrame(generalization_results)
original_f1 = generalization_df.loc[generalization_df.condition == "Original only", "f1"].iloc[0]
generalization_df["f1_gap_vs_original"] = (generalization_df["f1"] - original_f1).abs()
generalization_df

In [ ]:
TOLERANCE = 0.03
within_tolerance = (generalization_df["f1_gap_vs_original"] <= TOLERANCE).all()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=generalization_df, x="condition", y="f1", ax=axes[0], color="steelblue")
axes[0].set_title("F1 on Fixed Original Holdout")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=generalization_df, x="condition", y="roc_auc", ax=axes[1], color="indianred")
axes[1].set_title("ROC-AUC on Fixed Original Holdout")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "generalization_check.png"), dpi=100)
plt.show()

print(f"All conditions within {TOLERANCE} F1 of original-only:", within_tolerance)
print(
    "Staying within tolerance across all three conditions indicates the LLM-proposed severity "
    "parameters describe genuinely realistic regions of the drift space, rather than a distribution "
    "the original statistical generator would never have produced."
)

## **11 - Classical ML Model: Gradient Boosting and Random Forest**

### **11.1 - Overview**

This section trains the classical models that carry forward through the rest of the notebook, on the
full combined feature set from Sections 8 and 9 (raw plus human-engineered plus AI-suggested), using
the original train and test split established in Section 9.3.

In [ ]:
X_train_final = original_train[FULL_FEATURES]
y_train_final = original_train[TARGET]
X_test_final = original_test[FULL_FEATURES]
y_test_final = original_test[TARGET]

gbr_model = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
gbr_model.fit(X_train_final, y_train_final)

rf_model = RandomForestClassifier(n_estimators=400, max_depth=10, class_weight="balanced",
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train_final, y_train_final)

def score_model(model, name):
    preds = model.predict(X_test_final)
    probs = model.predict_proba(X_test_final)[:, 1]
    return {
        "model": name,
        "f1": f1_score(y_test_final, preds),
        "precision": precision_score(y_test_final, preds),
        "recall": recall_score(y_test_final, preds),
        "roc_auc": roc_auc_score(y_test_final, probs),
    }

classical_results = pd.DataFrame([
    score_model(gbr_model, "Gradient Boosting"),
    score_model(rf_model, "Random Forest"),
])
classical_results

### **11.2 - Selecting the Reference Classical Model**

Gradient boosting and random forest typically land within a small margin of each other on this task.
The model with the higher F1 score on the held-out test set is carried forward as the reference
classical model for the explainability and agentic sections.

In [ ]:
classical_model = gbr_model if classical_results.loc[0, "f1"] >= classical_results.loc[1, "f1"] else rf_model
classical_model_name = "Gradient Boosting" if classical_model is gbr_model else "Random Forest"
print("Reference classical model:", classical_model_name)

cm = confusion_matrix(y_test_final, classical_model.predict(X_test_final))
fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Stable", "Drifted"], yticklabels=["Stable", "Drifted"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"{classical_model_name}: Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "classical_confusion_matrix.png"), dpi=100)
plt.show()

## **12 - Light Deep Learning Model: PyTorch Feedforward Network**

### **12.1 - Overview**

This section trains a compact feedforward neural network on the same feature set and split as the
classical models, using standardized inputs and a device-agnostic setup that runs on CPU or GPU without
changes.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final.values)
X_test_scaled = scaler.transform(X_test_final.values)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(DEVICE)
y_train_t = torch.tensor(y_train_final.values, dtype=torch.float32).view(-1, 1).to(DEVICE)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(DEVICE)

class DriftNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

drift_net = DriftNet(X_train_t.shape[1]).to(DEVICE)
optimizer = torch.optim.Adam(drift_net.parameters(), lr=1e-3, weight_decay=1e-5)

pos_weight_value = (y_train_final == 0).sum() / max((y_train_final == 1).sum(), 1)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE))

EPOCHS = 200
BATCH_SIZE = 64
n_samples = X_train_t.shape[0]
history = []

for epoch in range(EPOCHS):
    perm = torch.randperm(n_samples)
    epoch_loss = 0.0
    for start in range(0, n_samples, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        xb, yb = X_train_t[idx], y_train_t[idx]

        optimizer.zero_grad()
        logits = drift_net(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.shape[0]

    history.append(epoch_loss / n_samples)
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS}, train BCE loss: {history[-1]:.5f}")

plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("Training BCE loss")
plt.title("DriftNet Training Curve")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "nn_training_curve.png"), dpi=100)
plt.show()

### **12.2 - Comparing the Neural Network to the Classical Model**

Predictions from the network are converted from logits to probabilities with a sigmoid, thresholded at
0.5, and scored on the same held-out test set as the classical models.

In [ ]:
drift_net.eval()
with torch.no_grad():
    nn_logits = drift_net(X_test_t).cpu().numpy().ravel()
nn_probs = 1 / (1 + np.exp(-nn_logits))
nn_preds = (nn_probs >= 0.5).astype(int)

nn_f1 = f1_score(y_test_final, nn_preds)
nn_auc = roc_auc_score(y_test_final, nn_probs)

model_comparison = pd.concat([
    classical_results[["model", "f1", "roc_auc"]],
    pd.DataFrame([{"model": "PyTorch FeedForward NN", "f1": nn_f1, "roc_auc": nn_auc}]),
], ignore_index=True)
model_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=model_comparison, x="model", y="f1", ax=ax, color="steelblue")
ax.set_title("F1 Score by Model")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "model_comparison.png"), dpi=100)
plt.show()

## **13 - Foundation Model Benchmark: TabPFN**

### **13.1 - Overview**

A pretrained tabular foundation model is trained once, on millions of synthetic tabular learning
tasks, and then applied to a new dataset directly, in a single forward pass, without a per-dataset
training loop. TabPFN is one such model, distributed as a small PyTorch checkpoint through the `tabpfn`
package, built for classification tasks with up to roughly 10,000 rows and 500 features. This batch-level
drift dataset, at around 1,200 batches and 16 features, sits comfortably inside that range.

The gradient boosting model in Section 11 and the feedforward network in Section 12 are both fit from
scratch on this dataset's training split, each learning its own decision boundary through an
optimization loop written and tuned for this task. TabPFN instead approximates Bayesian inference over
the training batches at inference time, using weights fixed during its original pretraining. This is a
distinct capability tier from a custom-trained model: no hyperparameter search, no training loop, and no
dataset-specific weight update, only a single fit call that conditions the frozen model on the training
batches and a predict call that reads off a classification for each test batch.

In [ ]:
TABPFN_AVAILABLE = False
try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
except ImportError:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=False)
        from tabpfn import TabPFNClassifier
        TABPFN_AVAILABLE = True
    except Exception as exc:
        print("TabPFN could not be installed, skipping the foundation-model benchmark. Error:", exc)
except Exception as exc:
    print("TabPFN import failed, skipping the foundation-model benchmark. Error:", exc)

if TABPFN_AVAILABLE:
    print("TabPFN imported successfully. Proceeding with the foundation-model benchmark.")
else:
    print("TabPFN unavailable in this environment. The classical ML and light DL comparison in "
          "Section 12 stands on its own, and this section's cells will report that no live "
          "foundation-model fit was run.")

### **13.2 - Fitting TabPFN on the Combined Feature Set**

TabPFN is fit on the same combined feature set used for the classical models and the light neural
network, the same 16 raw, human-engineered, and AI-suggested features from Sections 8 and 9, using the
same train and test split established in Section 9.3. No dataset-specific hyperparameter is tuned, and
no scaling step beyond what TabPFN applies internally is needed.

In [ ]:
tabpfn_f1 = None
tabpfn_auc = None

if TABPFN_AVAILABLE:
    try:
        tabpfn_model = TabPFNClassifier(device="cpu")
        tabpfn_model.fit(X_train_final.values, y_train_final.values)
        tabpfn_preds = tabpfn_model.predict(X_test_final.values)
        tabpfn_probs = tabpfn_model.predict_proba(X_test_final.values)[:, 1]
        tabpfn_f1 = f1_score(y_test_final, tabpfn_preds)
        tabpfn_auc = roc_auc_score(y_test_final, tabpfn_probs)
        print(f"TabPFN F1: {tabpfn_f1:.4f}, ROC-AUC: {tabpfn_auc:.4f}")
    except Exception as exc:
        print("TabPFN fit or predict failed, skipping the foundation-model row. Error:", exc)
        tabpfn_f1, tabpfn_auc = None, None
else:
    print("TabPFN not available in this environment. Skipping the foundation-model fit.")

### **13.3 - Comparing Classical ML, Light DL, and TabPFN**

The comparison below places TabPFN's F1 score on the same held-out test set alongside the reference
classical model from Section 11 and the feedforward network from Section 12. A model that was never fit
specifically to this drift-detection task landing in the same range as two models trained end to end on
it shows what a pretrained tabular foundation model buys a data engineer without a dataset-specific
training step: a usable classifier from a single fit call.

In [ ]:
foundation_comparison = model_comparison.copy()
if tabpfn_f1 is not None:
    foundation_comparison = pd.concat([
        foundation_comparison,
        pd.DataFrame([{"model": "TabPFN (foundation model)", "f1": tabpfn_f1, "roc_auc": tabpfn_auc}]),
    ], ignore_index=True)
    print("TabPFN row added to the model comparison.")
else:
    print("TabPFN row omitted from the comparison since no live fit was available in this environment.")

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=foundation_comparison, x="model", y="f1", ax=ax, color="steelblue")
ax.set_title("F1 Score: Classical ML vs. Light DL vs. TabPFN")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "foundation_model_comparison.png"), dpi=100)
plt.show()

foundation_comparison

## **14 - Explainability: LIME**

### **14.1 - Why LIME for This Case**

The other case studies in this series use permutation importance and partial dependence, and SHAP.
This notebook uses LIME (Local Interpretable Model-agnostic Explanations) instead, chosen deliberately
for the situation a data engineer is actually in: one specific batch has been flagged, and the question
is why that batch, not what matters on average across the whole pipeline. LIME is built exactly for
that question. It perturbs the neighborhood around one instance, fits a simple interpretable surrogate
model to the perturbed predictions, and reports which features drove that one prediction in
human-readable terms. A global importance ranking answers a different question, useful in its own
right, but not the one an on-call engineer opens a ticket to answer.

Section 14.3 aggregates LIME's local weights across several flagged batches to build a global picture
as well, which supports the law-rediscovery narrative in Section 15, but the headline method for this
case study is the local, per-batch explanation.

### **14.2 - Local Explanation for a Flagged Batch**

A LIME explainer is fit on the training feature distribution, then used to explain one specific
flagged batch, a true drifted batch the reference classical model scores with high confidence.

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_final.values,
    feature_names=FULL_FEATURES,
    class_names=["stable", "drifted"],
    mode="classification",
    discretize_continuous=True,
    random_state=42,
)

test_probs = classical_model.predict_proba(X_test_final)[:, 1]
candidate_idx = np.where((y_test_final.values == 1) & (test_probs > 0.8))[0]
flagged_row_pos = candidate_idx[0] if len(candidate_idx) > 0 else int(np.argmax(test_probs))

flagged_batch_id = original_test.iloc[flagged_row_pos]["batch_id"]
flagged_drift_type = original_test.iloc[flagged_row_pos]["drift_type"]
print("Flagged batch:", flagged_batch_id, "| true drift type:", flagged_drift_type,
      "| predicted probability drifted:", round(float(test_probs[flagged_row_pos]), 4))

local_explanation = explainer.explain_instance(
    X_test_final.values[flagged_row_pos], classical_model.predict_proba,
    num_features=10, num_samples=1000,
)

fig = local_explanation.as_pyplot_figure()
fig.set_size_inches(8, 5)
plt.title(f"LIME Local Explanation: {flagged_batch_id}")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lime_local_explanation.png"), dpi=100)
plt.show()

for feature_desc, weight in local_explanation.as_list():
    print(f"{weight:+.4f}  {feature_desc}")

### **14.3 - Aggregate LIME Importance Across Several Flagged Batches**

Averaging the absolute LIME weight for each feature across several confidently flagged batches gives a
global view built directly from the local explanations, rather than from a separate global-importance
algorithm. Each flagged batch is a different drift type where possible, so no single drift type
dominates the aggregate.

In [ ]:
flagged_positions = np.where((y_test_final.values == 1) & (test_probs > 0.7))[0][:30]

feature_to_bucket = {name: name for name in FULL_FEATURES}
aggregated_weights = {}
for pos in flagged_positions:
    exp = explainer.explain_instance(
        X_test_final.values[pos], classical_model.predict_proba, num_features=len(FULL_FEATURES),
        num_samples=500,
    )
    for feature_desc, weight in exp.as_list():
        for fname in FULL_FEATURES:
            if fname in feature_desc:
                aggregated_weights.setdefault(fname, []).append(abs(weight))
                break

lime_importance_df = pd.DataFrame({
    "feature": list(aggregated_weights.keys()),
    "mean_abs_lime_weight": [np.mean(v) for v in aggregated_weights.values()],
}).sort_values("mean_abs_lime_weight", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=lime_importance_df.head(12), x="mean_abs_lime_weight", y="feature", ax=ax, color="steelblue")
ax.set_title(f"Aggregate LIME Importance Across {len(flagged_positions)} Flagged Batches")
ax.set_xlabel("Mean |LIME weight|")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lime_aggregate_importance.png"), dpi=100)
plt.show()

lime_importance_df.head(12)

## **15 - The Law Rediscovery Moment: PSI and KS Statistic Dominate**

### **15.1 - What the Explanation Surfaces**

The aggregate LIME ranking in Section 14.3 places the aggregated PSI at or near the top, ahead of the
narrower, single-purpose features engineered in Section 8, the schema-mismatch flag, the max
cardinality delta, the row-count delta, and the total null count. The aggregated KS statistic also
ranks among the higher-importance features, alongside the max mean z-score, which is itself a
distance-style statistic rather than a superficial count. This is exactly the property the raw feature
set was designed to test for: PSI is the only feature in this dataset that responds to all four
injected drift types (it carries a dedicated missing-value bin and a categorical-proportion comparison,
in addition to the numeric decile comparison), while KS responds specifically to the two drift types
that change a numeric column's shape, mean shift and variance shift. Narrower features such as the
schema flag or the null-count total respond to only one drift type each, and rank lower as a result.

### **15.2 - Mapping the Result to the PSI Threshold Standard**

The cell below buckets every test-set batch by its aggregated PSI using the conventional thresholds
from Section 5.2, below 0.10, 0.10 to 0.25, and above 0.25, and reports the model's mean predicted
drift probability in each bucket. A model that had genuinely rediscovered the PSI convention, without
being told the threshold rule, would show predicted probability rising sharply across these three
buckets.

In [ ]:
def psi_bucket(value):
    if value < 0.10:
        return "Stable (PSI < 0.10)"
    elif value < 0.25:
        return "Moderate (0.10 to 0.25)"
    else:
        return "Significant (PSI > 0.25)"

diagnostic = original_test.copy()
diagnostic["predicted_prob_drifted"] = test_probs
diagnostic["psi_bucket"] = diagnostic["psi_agg"].apply(psi_bucket)

bucket_summary = diagnostic.groupby("psi_bucket")["predicted_prob_drifted"].agg(["mean", "count"])
bucket_summary = bucket_summary.reindex([
    "Stable (PSI < 0.10)", "Moderate (0.10 to 0.25)", "Significant (PSI > 0.25)"
])
print(bucket_summary.round(3))

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=diagnostic, x="psi_bucket", y="predicted_prob_drifted",
            order=["Stable (PSI < 0.10)", "Moderate (0.10 to 0.25)", "Significant (PSI > 0.25)"],
            ax=ax, color="indianred")
ax.set_title("Mean Predicted Drift Probability by PSI Threshold Bucket")
ax.set_xlabel("")
ax.set_ylabel("Mean predicted probability")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "psi_threshold_mapping.png"), dpi=100)
plt.show()

### **15.3 - The Historical Payoff**

The pattern recovered above is the same finding that let Kolmogorov and Smirnov's 1933 and 1948
distribution-distance test become, six decades later, the basis for a threshold rule credit-risk and
data-quality teams adopted without a machine learning model in the loop: a single distance number,
computed against a fixed reference, predicts population shift well enough to act on. This notebook
arrived at the same conclusion from the opposite direction, training a model on raw and engineered
statistics without ever stating the threshold rule, and finding that the model's own explanation of
its predictions converges on the aggregated PSI and KS statistic as the dominant signal. The 0.25
significant-drift boundary was never given to the model. It emerges from where the model's predicted
probability rises sharply in Section 15.2, in the same place the industry convention places it.

## **16 - Agentic Layer: LangGraph Data Quality Recommendation**

### **16.1 - Overview**

This section builds a small LangGraph graph with two nodes. The first node, `diagnose`, takes a
flagged batch's predicted drift probability, its aggregated PSI and KS statistic, and its top
LIME-contributing features, and assembles a plain-language diagnosis of the likely drift pattern. The
second node, `recommend`, sends that diagnosis to the free HF-hosted instruct model and asks for a
concrete recommendation an on-call data engineer could act on before the batch reaches the feature
store.

In [ ]:
from typing import TypedDict, Optional, List
from langgraph.graph import StateGraph, END

class BatchState(TypedDict):
    batch_id: str
    predicted_prob: float
    top_features: List[str]
    psi_agg: float
    ks_stat_agg: float
    mean_zscore_max: float
    var_zscore_max: float
    schema_mismatch_flag: float
    null_rate_delta: float
    diagnosis: Optional[str]
    recommendation: Optional[str]

def diagnose_node(state: BatchState) -> BatchState:
    prob = state["predicted_prob"]
    top_feats = ", ".join(state["top_features"])

    if state["schema_mismatch_flag"] >= 1:
        pattern = "a schema drift, a new categorical value outside the reference schema"
    elif state["null_rate_delta"] > 0.05:
        pattern = "a null-rate spike, an upstream field silently failing to populate"
    elif state["var_zscore_max"] > state["mean_zscore_max"] and state["var_zscore_max"] > 2:
        pattern = "a variance shift, the spread of a numeric column widening or narrowing"
    elif state["mean_zscore_max"] > 2:
        pattern = "a mean shift, a numeric column's central tendency moving away from baseline"
    else:
        pattern = "a general distributional shift not clearly tied to one mechanism"

    diagnosis = (
        f"Batch {state['batch_id']} was flagged with predicted drift probability {prob:.2f}. "
        f"Aggregated PSI is {state['psi_agg']:.3f} and aggregated KS statistic is "
        f"{state['ks_stat_agg']:.3f}. The pattern most resembles {pattern}. The top "
        f"LIME-contributing features for this batch were: {top_feats}."
    )
    state["diagnosis"] = diagnosis
    return state

def recommend_node(state: BatchState) -> BatchState:
    prompt = f"""You are a data engineering assistant. Given this diagnosis of a flagged data batch,
write a 3 to 4 sentence recommendation in plain language for the on-call data engineer. Recommend a
specific, concrete action appropriate to the pattern described (for example: pause the batch before it
reaches the feature store and investigate the upstream source column, roll back a recent schema or
pipeline change, or open a ticket with the source system owner), and note the confidence implied by the
predicted probability and the PSI value.

Diagnosis: {{state['diagnosis']}}
"""
    llm_text = call_hf_llm(prompt, max_tokens=250, temperature=0.5)
    if llm_text is None:
        llm_text = (
            "HF router unavailable, using fallback recommendation. Based on the diagnosis, pause this "
            "batch before it reaches the feature store, notify the owner of the upstream source table, "
            "and compare the batch's schema and null rates against the last known-good extract before "
            "allowing subsequent batches through unchecked."
        )
    state["recommendation"] = llm_text
    return state

graph = StateGraph(BatchState)
graph.add_node("diagnose", diagnose_node)
graph.add_node("recommend", recommend_node)
graph.set_entry_point("diagnose")
graph.add_edge("diagnose", "recommend")
graph.add_edge("recommend", END)
drift_agent = graph.compile()

print("LangGraph agent compiled with nodes: diagnose -> recommend")

### **16.2 - Running the Agent on the Flagged Batch**

The graph is invoked on the same flagged test-set batch used for the LIME local explanation in Section
13.2.

In [ ]:
top_feature_names = lime_importance_df.head(5)["feature"].tolist()

sample_row = original_test.iloc[flagged_row_pos]
sample_state: BatchState = {
    "batch_id": sample_row["batch_id"],
    "predicted_prob": float(test_probs[flagged_row_pos]),
    "top_features": top_feature_names,
    "psi_agg": float(sample_row["psi_agg"]),
    "ks_stat_agg": float(sample_row["ks_stat_agg"]),
    "mean_zscore_max": float(batches_feat.loc[sample_row.name, "mean_zscore_max"]),
    "var_zscore_max": float(batches_feat.loc[sample_row.name, "var_zscore_max"]),
    "schema_mismatch_flag": float(batches_feat.loc[sample_row.name, "schema_mismatch_flag"]),
    "null_rate_delta": float(batches_feat.loc[sample_row.name, "null_rate_delta"]),
    "diagnosis": None,
    "recommendation": None,
}

result_state = drift_agent.invoke(sample_state)
print("Diagnosis:\n", result_state["diagnosis"])
print("\nRecommendation:\n", result_state["recommendation"])

### **16.3 - From Fixed Pipeline to Autonomous Agent**

The `diagnose_node` above decides the drift pattern with a hardcoded if/elif chain over four
thresholds, written once in Python and never revisited by a model. It calls no tool beyond string
formatting, and it carries no memory of any batch flagged before it. Every invocation reasons about the
batch in front of it and nothing else. This is a fixed pipeline: two nodes, one path, the same shape of
reasoning applied to every batch regardless of what that specific batch actually needs checked.

The agent built below is autonomous on three axes: planning, tool use, and memory. On planning, an LLM
decides which of four available functions to call and in what order, instead of following a hardcoded
if/elif chain, and a conditional edge routes control back to the same node until the model stops
requesting tools and produces a final answer, at which point the graph proceeds to `END`. On tool use,
the model calls real Python functions, `compute_batch_stats`, `predict_drift_label`,
`recall_similar_incidents`, and `flag_for_pipeline_owner`, instead of having its diagnosis assembled by
string formatting inside a single function. On memory, each session is appended to an append-only JSONL
log, so a later call to `recall_similar_incidents` can surface a prior batch with a similar PSI and
z-score profile and reference it as precedent, a capability the fixed pipeline above has no mechanism
for, since it treats every batch as first contact.

### **16.4 - Tools and Memory for the Autonomous Agent**

Four tools are exposed to the model: `compute_batch_stats` returns the PSI, KS, and z-score breakdown
for a batch, `predict_drift_label` calls the trained reference classical model, `recall_similar_incidents`
searches the append-only memory log for the most similar past flagged batches by normalized distance over
the same statistics, and `flag_for_pipeline_owner` is the terminal action a data engineer would actually
take, escalating the batch with a stated reason. The memory log is a plain JSON-lines file written at
runtime to `data/agent_memory.jsonl`, one line per past session, so no vector database or external
service is required to give the agent access to precedent.

A larger instruct model than the one used for the GenAI feature and augmentation steps in Sections 9
and 10 is used here. Reliable JSON-schema tool calling asks more of a model than free-text generation
does, so this section uses `Qwen/Qwen2.5-7B-Instruct` rather than the 1.5B-parameter variant used
elsewhere in the notebook, while staying on the same free Hugging Face router client already configured
in Section 6.

In [ ]:
AGENT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
AGENT_MEMORY_PATH = os.path.join(DATA_DIR, "agent_memory.jsonl")

# Runtime registry so the same tools can answer questions about a batch that was never part of the
# original test set, for example a batch entered by hand in Section 17's interactive demo.
RUNTIME_BATCH_REGISTRY = {}


def compute_batch_stats(batch_id):
    """Return the PSI, KS statistic, and z-score breakdown for a batch, whether it came from the
    evaluated test set or was registered at runtime by the interactive demo."""
    if batch_id in RUNTIME_BATCH_REGISTRY:
        return RUNTIME_BATCH_REGISTRY[batch_id]["stats"]
    match = original_test[original_test["batch_id"] == batch_id]
    if match.empty:
        return {"error": f"Batch {batch_id} not found."}
    row = match.iloc[0]
    feat_row = batches_feat.loc[row.name]
    return {
        "batch_id": batch_id,
        "psi_agg": round(float(row["psi_agg"]), 4),
        "ks_stat_agg": round(float(row["ks_stat_agg"]), 4),
        "mean_zscore_max": round(float(feat_row["mean_zscore_max"]), 4),
        "var_zscore_max": round(float(feat_row["var_zscore_max"]), 4),
        "null_rate_delta": round(float(feat_row["null_rate_delta"]), 4),
        "cardinality_delta_max": round(float(feat_row["cardinality_delta_max"]), 4),
        "schema_mismatch_flag": int(feat_row["schema_mismatch_flag"]),
    }


def predict_drift_label(batch_id):
    """Call the trained reference classical model and return its prediction for this batch."""
    if batch_id in RUNTIME_BATCH_REGISTRY:
        return RUNTIME_BATCH_REGISTRY[batch_id]["prediction"]
    match = original_test[original_test["batch_id"] == batch_id]
    if match.empty:
        return {"error": f"Batch {batch_id} not found."}
    row_pos = original_test.index.get_loc(match.index[0])
    prob = float(test_probs[row_pos])
    label = "drifted" if prob >= 0.5 else "stable"
    return {"batch_id": batch_id, "predicted_label": label, "predicted_probability": round(prob, 4)}


def _load_agent_memory():
    if not os.path.exists(AGENT_MEMORY_PATH):
        return []
    records = []
    with open(AGENT_MEMORY_PATH, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records


def recall_similar_incidents(batch_id, n=3):
    """Return the n most similar past flagged batches logged in agent memory, by normalized Euclidean
    distance over PSI, KS, and z-score features. No vector database is needed at this scale."""
    records = _load_agent_memory()
    if not records:
        return {"message": "No prior agent sessions logged yet. This is the first incident on record."}

    stats = compute_batch_stats(batch_id)
    if "error" in stats:
        return stats

    keys = ["psi_agg", "ks_stat_agg", "mean_zscore_max", "var_zscore_max", "null_rate_delta"]
    query_vec = np.array([stats[k] for k in keys])
    all_vecs = [np.array([rec["stats"].get(k, 0.0) for k in keys]) for rec in records] + [query_vec]
    scales = np.max(np.abs(np.vstack(all_vecs)), axis=0)
    scales = np.where(scales == 0, 1.0, scales)

    scored = []
    for rec in records:
        rec_vec = np.array([rec["stats"].get(k, 0.0) for k in keys])
        dist = float(np.linalg.norm((query_vec - rec_vec) / scales))
        scored.append((dist, rec))
    scored.sort(key=lambda pair: pair[0])

    top = [
        {"batch_id": rec["batch_id"], "diagnosis": rec.get("diagnosis"),
         "recommendation": rec.get("recommendation"), "similarity_distance": round(dist, 4)}
        for dist, rec in scored[:n]
    ]
    return {"similar_incidents": top}


def flag_for_pipeline_owner(reason):
    """Terminal action: record that this batch was escalated to the pipeline owner, with a reason."""
    return {
        "status": "flagged",
        "reason": reason,
        "note": "The pipeline owner has been notified. This batch should be held before it reaches "
                "the feature store.",
    }


def log_agent_session(batch_id, stats, predicted_label, diagnosis, recommendation):
    """Append this session's outcome to the append-only JSONL memory file."""
    record = {
        "batch_id": batch_id,
        "stats": stats,
        "predicted_label": predicted_label,
        "diagnosis": diagnosis,
        "recommendation": recommendation,
    }
    with open(AGENT_MEMORY_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")
    return record


AGENT_TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "compute_batch_stats",
        "description": "Return the PSI, KS statistic, and z-score breakdown for a flagged batch.",
        "parameters": {
            "type": "object",
            "properties": {
                "batch_id": {"type": "string", "description": "The batch identifier, e.g. BATCH_00042."},
            },
            "required": ["batch_id"],
        },
    }},
    {"type": "function", "function": {
        "name": "predict_drift_label",
        "description": "Call the trained reference classical model and return its predicted label and "
                        "probability for this batch.",
        "parameters": {
            "type": "object",
            "properties": {
                "batch_id": {"type": "string", "description": "The batch identifier, e.g. BATCH_00042."},
            },
            "required": ["batch_id"],
        },
    }},
    {"type": "function", "function": {
        "name": "recall_similar_incidents",
        "description": "Look up the most similar past flagged batches logged in agent memory, by "
                        "distance over PSI, KS, and z-score features.",
        "parameters": {
            "type": "object",
            "properties": {
                "batch_id": {"type": "string", "description": "The batch identifier to find precedent for."},
                "n": {"type": "integer", "description": "Number of similar incidents to return.", "default": 3},
            },
            "required": ["batch_id"],
        },
    }},
    {"type": "function", "function": {
        "name": "flag_for_pipeline_owner",
        "description": "Terminal action. Escalate this batch to the pipeline owner with a stated reason, "
                        "holding it before it reaches the feature store.",
        "parameters": {
            "type": "object",
            "properties": {
                "reason": {"type": "string", "description": "A short, specific reason for the escalation."},
            },
            "required": ["reason"],
        },
    }},
]

AGENT_TOOL_REGISTRY = {
    "compute_batch_stats": compute_batch_stats,
    "predict_drift_label": predict_drift_label,
    "recall_similar_incidents": recall_similar_incidents,
    "flag_for_pipeline_owner": flag_for_pipeline_owner,
}

print("Tools registered:", list(AGENT_TOOL_REGISTRY.keys()))
print("Agent memory file:", AGENT_MEMORY_PATH)

### **16.5 - Building the ReAct Loop with Conditional Edges**

The graph below has the same two roles as any tool-using ReAct agent, an `agent` node that calls the
LLM with tool-calling enabled, and a `tools` node that executes whatever tool calls the LLM requested.
A conditional edge inspects the LLM's latest response after every `agent` step: if it contains tool
calls, control routes to `tools` and then back to `agent` so the model can read the tool results and
decide its next move, and if it contains no tool calls, control routes to `END`. The model decides how many rounds of
investigation a given batch needs, based on what the tool results show at each step.

In [ ]:
class AgentState(TypedDict):
    messages: List[dict]


def agent_node(state: AgentState) -> AgentState:
    if hf_client is None:
        state["messages"].append({
            "role": "assistant",
            "content": "HF router unavailable, no HF_TOKEN configured. The fixed pipeline's diagnosis "
                       "above is the only recommendation available in this environment.",
        })
        return state
    try:
        response = hf_client.chat.completions.create(
            model=AGENT_MODEL,
            messages=state["messages"],
            tools=AGENT_TOOL_SCHEMAS,
            tool_choice="auto",
            max_tokens=500,
            temperature=0.2,
        )
        message = response.choices[0].message
        state["messages"].append(message.model_dump(exclude_none=True))
    except Exception as exc:
        state["messages"].append({
            "role": "assistant",
            "content": f"Tool-calling request failed ({exc}). Stopping the autonomous loop.",
        })
    return state


def tools_node(state: AgentState) -> AgentState:
    last_message = state["messages"][-1]
    for tool_call in last_message.get("tool_calls", []) or []:
        fn_name = tool_call["function"]["name"]
        try:
            fn_args = json.loads(tool_call["function"]["arguments"])
        except json.JSONDecodeError:
            fn_args = {}
        fn = AGENT_TOOL_REGISTRY.get(fn_name)
        if fn is None:
            result = {"error": f"Unknown tool {fn_name}"}
        else:
            try:
                result = fn(**fn_args)
            except Exception as exc:
                result = {"error": str(exc)}
        state["messages"].append({
            "role": "tool",
            "tool_call_id": tool_call.get("id", fn_name),
            "name": fn_name,
            "content": json.dumps(result),
        })
    return state


def route_after_agent(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if last_message.get("tool_calls"):
        return "tools"
    return "end"


autonomous_drift_agent = None
try:
    agent_graph = StateGraph(AgentState)
    agent_graph.add_node("agent", agent_node)
    agent_graph.add_node("tools", tools_node)
    agent_graph.set_entry_point("agent")
    agent_graph.add_conditional_edges("agent", route_after_agent, {"tools": "tools", "end": END})
    agent_graph.add_edge("tools", "agent")
    autonomous_drift_agent = agent_graph.compile()
    print("Autonomous ReAct-style agent compiled with nodes: agent <-> tools, conditional routing to END.")
except Exception as exc:
    print("Could not compile the autonomous agent graph, the fixed pipeline above remains the only "
          "agentic path available. Error:", exc)

### **16.6 - Running the Autonomous Agent on the Flagged Batch**

The autonomous agent is pointed at the same flagged batch used for the fixed pipeline above and for
the LIME local explanation in Section 14.2. Its transcript is printed tool call by tool call, so the
planning the LLM does, and not just its final answer, is visible. The session is then appended to the
JSONL memory file, so a future call to `recall_similar_incidents` can cite this batch as precedent.

In [ ]:
AGENT_SYSTEM_PROMPT = (
    "You are a data quality monitoring agent for a machine learning feature pipeline. A batch has been "
    "flagged by an upstream classifier. Investigate it using the tools available to you: "
    "compute_batch_stats to see its PSI, KS, and z-score breakdown, predict_drift_label to confirm the "
    "model's prediction, and recall_similar_incidents to check whether a similar pattern has been seen "
    "before. Once you have reached a conclusion, call flag_for_pipeline_owner exactly once, as your "
    "final step, stating your reasoning as the escalation reason."
)

if autonomous_drift_agent is not None:
    initial_state: AgentState = {
        "messages": [
            {"role": "system", "content": AGENT_SYSTEM_PROMPT},
            {"role": "user", "content": f"Investigate batch {flagged_batch_id}."},
        ]
    }
    try:
        final_state = autonomous_drift_agent.invoke(initial_state, {"recursion_limit": 12})
        for msg in final_state["messages"]:
            role = msg.get("role")
            if role == "assistant" and msg.get("tool_calls"):
                for tc in msg["tool_calls"]:
                    print(f"[agent -> tool] {tc['function']['name']}({tc['function']['arguments']})")
            elif role == "tool":
                print(f"[tool -> agent] {msg['name']}: {msg['content']}")
            elif role == "assistant" and msg.get("content"):
                print(f"[agent] {msg['content']}")

        final_assistant_messages = [
            m["content"] for m in final_state["messages"]
            if m.get("role") == "assistant" and m.get("content")
        ]
        final_summary = final_assistant_messages[-1] if final_assistant_messages else (
            "No final summary produced."
        )
        log_agent_session(
            batch_id=flagged_batch_id,
            stats=compute_batch_stats(flagged_batch_id),
            predicted_label=predict_drift_label(flagged_batch_id).get("predicted_label"),
            diagnosis="Autonomous agent investigation, see transcript above.",
            recommendation=final_summary,
        )
        print("\nSession logged to", AGENT_MEMORY_PATH)
    except Exception as exc:
        print("Autonomous agent run failed, no live HF tool-calling session available. Error:", exc)
else:
    print("Autonomous agent not available in this environment. The fixed pipeline above remains the "
          "only agentic path for this run.")

## **17 - Interactive Prediction Demo**

### **17.1 - Overview**

The function below ties the full pipeline together for a single candidate batch. It takes exactly the
batch-level summary numbers a data engineer would actually have on hand after computing the comparison
against the reference (the null-rate delta, row-count delta, max mean and variance z-scores, max
cardinality delta, the schema-mismatch flag, the aggregated KS statistic, and the aggregated PSI), fills
in the remaining raw statistics with neutral reference-baseline values since a per-column breakdown was
not supplied, computes the AI-suggested features from that row, predicts drifted or stable, runs a LIME
explanation to find the top local contributors, and calls the LangGraph agent for a recommendation.

In [ ]:
def predict_batch(null_rate_delta, row_count_delta, mean_zscore_max, var_zscore_max,
                   cardinality_delta_max, schema_mismatch_flag, ks_stat_agg, psi_agg,
                   batch_id="USER-INPUT-BATCH", use_autonomous_agent=False):
    row = {
        "row_count": REFERENCE_BATCH_SIZE + row_count_delta,
        "null_count_total": (BASE_NULL_RATE + null_rate_delta) * (REFERENCE_BATCH_SIZE + row_count_delta),
        "ks_stat_agg": ks_stat_agg,
        "psi_agg": psi_agg,
        "null_rate_delta": null_rate_delta,
        "row_count_delta": row_count_delta,
        "mean_zscore_max": mean_zscore_max,
        "var_zscore_max": var_zscore_max,
        "cardinality_delta_max": cardinality_delta_max,
        "schema_mismatch_flag": schema_mismatch_flag,
    }
    row_df = pd.DataFrame([row])
    row_df, _ = build_ai_features(row_df, ai_feature_specs)

    predicted_prob = float(classical_model.predict_proba(row_df[FULL_FEATURES])[:, 1][0])
    predicted_label = "drifted" if predicted_prob >= 0.5 else "stable"

    row_explanation = explainer.explain_instance(
        row_df[FULL_FEATURES].values[0], classical_model.predict_proba, num_features=5, num_samples=500,
    )
    top_features_local = []
    for feature_desc, _ in row_explanation.as_list():
        for fname in FULL_FEATURES:
            if fname in feature_desc:
                top_features_local.append(fname)
                break

    # Register this batch at runtime so the autonomous agent's tools (Section 16.4) can look it up by
    # batch_id even though it never went through the original train and test split.
    RUNTIME_BATCH_REGISTRY[batch_id] = {
        "stats": {
            "batch_id": batch_id, "psi_agg": psi_agg, "ks_stat_agg": ks_stat_agg,
            "mean_zscore_max": mean_zscore_max, "var_zscore_max": var_zscore_max,
            "null_rate_delta": null_rate_delta, "cardinality_delta_max": cardinality_delta_max,
            "schema_mismatch_flag": schema_mismatch_flag,
        },
        "prediction": {
            "batch_id": batch_id, "predicted_label": predicted_label,
            "predicted_probability": round(predicted_prob, 4),
        },
    }

    print(f"Predicted label: {predicted_label} (probability drifted: {predicted_prob:.3f})")
    print(f"Top contributing features: {top_features_local}")

    if use_autonomous_agent and autonomous_drift_agent is not None:
        initial_state: AgentState = {
            "messages": [
                {"role": "system", "content": AGENT_SYSTEM_PROMPT},
                {"role": "user", "content": f"Investigate batch {batch_id}."},
            ]
        }
        try:
            final_state = autonomous_drift_agent.invoke(initial_state, {"recursion_limit": 12})
            final_assistant_messages = [
                m["content"] for m in final_state["messages"]
                if m.get("role") == "assistant" and m.get("content")
            ]
            final_summary = final_assistant_messages[-1] if final_assistant_messages else (
                "No final summary produced."
            )
            log_agent_session(
                batch_id=batch_id, stats=RUNTIME_BATCH_REGISTRY[batch_id]["stats"],
                predicted_label=predicted_label,
                diagnosis="Autonomous agent investigation, interactive demo.",
                recommendation=final_summary,
            )
            print(f"\nAutonomous agent conclusion: {final_summary}")
        except Exception as exc:
            print("Autonomous agent run failed, falling back to the fixed pipeline. Error:", exc)
            use_autonomous_agent = False

    if not use_autonomous_agent or autonomous_drift_agent is None:
        state: BatchState = {
            "batch_id": batch_id,
            "predicted_prob": predicted_prob,
            "top_features": top_features_local,
            "psi_agg": psi_agg,
            "ks_stat_agg": ks_stat_agg,
            "mean_zscore_max": mean_zscore_max,
            "var_zscore_max": var_zscore_max,
            "schema_mismatch_flag": schema_mismatch_flag,
            "null_rate_delta": null_rate_delta,
            "diagnosis": None,
            "recommendation": None,
        }
        result = drift_agent.invoke(state)
        print(f"\nDiagnosis: {result['diagnosis']}")
        print(f"\nRecommendation: {result['recommendation']}")
        return result

    return RUNTIME_BATCH_REGISTRY[batch_id]


_ = predict_batch(
    null_rate_delta=0.005, row_count_delta=-40, mean_zscore_max=8.5, var_zscore_max=0.3,
    cardinality_delta_max=0, schema_mismatch_flag=0, ks_stat_agg=0.42, psi_agg=1.35,
    batch_id="DEMO-SEVERE-MEAN-SHIFT",
)

In [ ]:
# A second example, a batch with a mild schema drift that should read as a moderate, not severe, case.
_ = predict_batch(
    null_rate_delta=0.01, row_count_delta=15, mean_zscore_max=0.4, var_zscore_max=0.2,
    cardinality_delta_max=1, schema_mismatch_flag=1, ks_stat_agg=0.05, psi_agg=0.18,
    batch_id="DEMO-MILD-SCHEMA-DRIFT",
)

### **17.2 - Trying the Autonomous Agent Variant**

Passing `use_autonomous_agent=True` routes the same candidate batch through the ReAct-style agent
from Section 16.4 instead of the fixed pipeline, letting the model decide for itself whether to check
batch statistics, confirm the prediction, or look for a similar past incident before escalating.

In [ ]:
_ = predict_batch(
    null_rate_delta=0.01, row_count_delta=15, mean_zscore_max=0.4, var_zscore_max=0.2,
    cardinality_delta_max=1, schema_mismatch_flag=1, ks_stat_agg=0.05, psi_agg=0.18,
    batch_id="DEMO-MILD-SCHEMA-DRIFT-AUTONOMOUS", use_autonomous_agent=True,
)

## **18 - Conclusion and Takeaways**

### **18.1 - Conclusion**

This notebook built a data-quality drift classification pipeline from a synthetic batch dataset, with
drift injected across four mechanisms, mean shift, variance shift, null-rate spike, and schema drift, at
three severities each. A gradient boosting classifier, a light PyTorch feedforward network, and a
pretrained TabPFN foundation model were compared on a combined raw, human-engineered, and AI-suggested
feature set, with the combined feature set outperforming either engineered source alone and TabPFN
matching the two custom-trained models without a dataset-specific training step. LIME explanations,
computed per batch and aggregated across several flagged batches, surfaced the aggregated PSI and the
aggregated Kolmogorov-Smirnov statistic as the dominant drivers of the drift decision, ahead of narrower
single-purpose features. That ranking, together with the model's rising predicted probability across the
conventional PSI threshold buckets, recovers the same PSI and KS-based standard data-quality and
credit-risk teams have relied on since the 1990s, without the model ever being told the threshold rule.
A fixed LangGraph pipeline turned a flagged batch's prediction and top contributing features into a
plain-language recommendation, and an autonomous ReAct-style agent built alongside it added planning,
real tool use, and a persistent memory of past incidents, letting the model decide for itself what to
check before it escalates a batch to the pipeline owner.

### **18.2 - Takeaways**

- Four drift mechanisms with almost no shared surface-level signature, a mean shift, a variance shift, a
  null-rate spike, and a schema drift, are all still caught by the same two aggregated statistical
  measures, because PSI and the KS statistic measure distributional distance rather than any one
  mechanism specifically.
- The PSI threshold convention (below 0.10 stable, 0.10 to 0.25 moderate, above 0.25 significant),
  adopted in credit risk in the 1990s decades before it reached ML feature monitoring, reappeared in
  this notebook's model behavior without being specified anywhere in training.
- LIME's local, per-instance explanations, aggregated across several flagged batches, recovered the same
  global picture a dedicated global-importance method would, while still answering the question an
  on-call engineer actually asks about one specific flagged batch.
- Superficial features a data engineer would reflexively compute, row-count delta, a single
  schema-mismatch boolean, cardinality delta, ranked well below the aggregated distance measures,
  because each targets only one drift mechanism while PSI and KS respond across several.
- A language model proposing feature transforms and augmentation parameters, rather than raw numbers,
  kept the synthetic augmentation grounded in the same generation logic used to build the original
  dataset, and the three-way generalization check confirmed the augmented regions were realistic rather
  than out of distribution.
- TabPFN, fit once on the training batches with no hyperparameter search, reached a comparable F1 score
  to the gradient boosting classifier and the feedforward network, showing what a pretrained tabular
  foundation model buys a data engineer over a model trained from scratch for this one task.
- The fixed LangGraph pipeline hardcoded its diagnosis logic in Python if/elif branches, called no tools,
  and kept no memory across runs. The autonomous agent replaced that with an LLM that plans its own
  sequence of tool calls, reads real batch statistics and past incidents through those tools, and logs
  each session to an append-only file so a later batch can be diagnosed with reference to precedent.